In [ ]:
import os
import numpy as np
import evaluate
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    pipeline
)
from datasets import Dataset, DatasetDict
import pandas as pd

emotion_map = {
    'joyful': 'joy', 'excited': 'joy', 'proud': 'joy', 'hopeful': 'joy', 'confident': 'joy', 'content': 'joy', 'prepared': 'joy', 'anticipating': 'joy',
    'sad': 'sadness', 'lonely': 'sadness', 'nostalgic': 'sadness', 'disappointed': 'sadness', 'devastated': 'sadness', 'grieving': 'sadness',
    'angry': 'anger', 'annoyed': 'anger', 'furious': 'anger', 'jealous': 'anger', 'disgusted': 'anger',
    'afraid': 'fear', 'terrified': 'fear', 'anxious': 'fear', 'apprehensive': 'fear',
    'surprised': 'surprise', 'impressed': 'surprise',
    'guilty': 'guilt', 'ashamed': 'guilt', 'embarrassed': 'guilt',
    'caring': 'love', 'trusting': 'love', 'faithful': 'love', 'sentimental': 'love'
}

DATA_PROCESSED_PATH = '../data/processed/empatheticdialogues'

df_train = pd.read_csv(DATA_PROCESSED_PATH+ '/train.csv', on_bad_lines='skip')
df_valid = pd.read_csv(DATA_PROCESSED_PATH+ '/valid.csv', on_bad_lines='skip')
df_test = pd.read_csv(DATA_PROCESSED_PATH+ '/test.csv', on_bad_lines='skip')

# Применяем маппинг ко всем выборкам
df_train['context'] = df_train['context'].map(emotion_map)
df_valid['context'] = df_valid['context'].map(emotion_map)
df_test['context'] = df_test['context'].map(emotion_map)

# Удаляем строки, если какие-то эмоции не смапились (на всякий случай)
df_train = df_train.dropna(subset=['context'])
df_valid = df_valid.dropna(subset=['context'])
df_test = df_test.dropna(subset=['context'])

emotions = df_train['context'].unique().tolist()

# Исправляем извлечение уникальных эмоций
emotions = df_train['context'].unique().tolist()
label2id = {emotion: i for i, emotion in enumerate(emotions)}
id2label = {i: emotion for i, emotion in enumerate(emotions)}

print(f"Размер выборки: train={len(df_train)}, valid={len(df_valid)}")
print(f"Всего эмоций: {len(emotions)}")

# Преобразуем pandas DataFrame в Hugging Face Dataset
dataset = DatasetDict({
    "train": Dataset.from_pandas(df_train),
    "validation": Dataset.from_pandas(df_valid),
    "test": Dataset.from_pandas(df_test)
})


Размер выборки: train=74181, valid=6146
Всего эмоций: 7


In [ ]:
#  Токенизация текста
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    inputs = tokenizer(examples["utterance"], padding="max_length", truncation=True, max_length=128)

    inputs["label"] = [label2id[c] for c in examples["context"]]
    return inputs

print("Токенизация данных...")
tokenized_datasets = dataset.map(preprocess_function, batched=True)

print("Инициализация модели...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=len(emotions), 
    id2label=id2label, 
    label2id=label2id
)


Токенизация данных...


Map: 100%|██████████| 5480/5480 [00:00<00:00, 31026.06 examples/s]
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Инициализация модели...


In [ ]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)


training_args = TrainingArguments(
    output_dir="./emotion_classifier", # Папка для сохранения модели
    eval_strategy="epoch",       # Оценка в конце каждой эпохи
    save_strategy="epoch",             # Сохранение в конце каждой эпохи
    learning_rate=2e-5,                # Скорость обучения
    per_device_train_batch_size=16,    # Размер батча для обучения
    per_device_eval_batch_size=16,     # Размер батча для оценки
    num_train_epochs=3,                # Количество эпох
    weight_decay=0.01,
    load_best_model_at_end=True,       # Загрузить лучшую модель в конце
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

print("Начало обучения (это может занять некоторое время)...")
trainer.train()


Начало обучения (это может занять некоторое время)...


Epoch,Training Loss,Validation Loss,Accuracy
1,1.300100,1.148279,0.593069
2,1.124800,1.130346,0.606085
3,1.006100,1.168901,0.608851


TrainOutput(global_step=13911, training_loss=1.171433193833932, metrics={'train_runtime': 990.5851, 'train_samples_per_second': 224.658, 'train_steps_per_second': 14.043, 'total_flos': 7370580235352832.0, 'train_loss': 1.171433193833932, 'epoch': 3.0})

In [ ]:
print("Сохранение модели...")
trainer.save_model("./my_emotion_model")
tokenizer.save_pretrained("./my_emotion_model")

print("Тестирование модели на новых данных!")
# Создаем пайплайн для быстрой классификации
classifier = pipeline("text-classification", model="./my_emotion_model", tokenizer="./my_emotion_model")

test_texts = [
    "I just got a promotion at work! I can't believe it!",
    "My dog passed away yesterday. I miss him so much.",
    "Walking home alone in the dark alley made my heart race.",
    "I am so mad at my roommate for eating my food again!"
]

for text in test_texts:
    result = classifier(text)[0]
    print(f"Текст: '{text}'")
    print(f"Эмоция: {result['label']} (Уверенность: {result['score']:.4f})\n")

Сохранение модели...


Device set to use cuda:0


Тестирование модели на новых данных!
Текст: 'I just got a promotion at work! I can't believe it!'
Эмоция: surprise (Уверенность: 0.6183)

Текст: 'My dog passed away yesterday. I miss him so much.'
Эмоция: sadness (Уверенность: 0.7555)

Текст: 'Walking home alone in the dark alley made my heart race.'
Эмоция: fear (Уверенность: 0.8838)

Текст: 'I am so mad at my roommate for eating my food again!'
Эмоция: anger (Уверенность: 0.9598)

